# Deep Learning for Media
#### MPATE-GE 2039 - DM-GY 9103

---

## VR Chinese OCR Evaluation
### Robustness Analysis of Vision Models for Handwritten Chinese Character Recognition


## 1. Setup
Run this cell first — it adds the repo root to the Python path so all modules are importable.

In [9]:
# Add repo root to sys.path so imports work when running from /notebooks/
import sys, importlib
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import data.utils as data_utils
import utils.utils as utils_facade
import perturbations.perturbation_analysis as pa
import utils as u

importlib.reload(data_utils)
importlib.reload(utils_facade)
importlib.reload(pa)
importlib.reload(u)

# sanity check
import inspect
print(u.load_data.__module__, inspect.getsourcefile(u.load_data), u.load_data.__code__.co_firstlineno)


data.utils /Users/ek/Documents/GitHub/vr-chinese-ocr-eval/data/utils.py 189


In [10]:
# Fix random seeds for reproducibility across numpy and tensorflow
from numpy.random import seed
seed(124)
import tensorflow as tf
tf.keras.utils.set_random_seed(124)

## 2. Load Dataset & Apply Perturbations

**Dataset:** CASIA-HWDB2-line (offline handwritten Chinese) — https://huggingface.co/datasets/Teklia/CASIA-HWDB2-line

**Data split:**
- 1500 clean images sampled from train split
- Perturbation pool drawn from a *disjoint* set of indices (no overlap with clean)
- Final split: 80% train / 20% test
- Train further split: 80% train / 20% val

**Perturbation count:**
- From our presentation: ~1000–1500 clean + 200–300 perturbed
- Revised: 1500 clean + 9 types × 50 min per type = 450 perturbed
- Each image receives exactly one perturbation type (no combinations)

**Perturbation types (9 total):**

Lighting conditions:
- `radial_glare` — spotlight-style glare reflecting off the whiteboard surface
- `light_streak` — directional light band across the image
- `contrast_variation` — under/overexposed lighting in the room
- `white_balance_shift` — Quest auto white balance tinting the scene warm or cool

Camera / capture quality:
- `gaussian_noise` — sensor and camera noise
- `motion_blur` — camera shake or hand movement while writing
- `low_resolution` — degraded quality from the Meta Horizon cast stream

Geometric distortion:
- `perspective_warp` — camera not perfectly frontal to the whiteboard

Physical obstruction:
- `occlusion` — a rectangular block masking part of the character (e.g. hand in frame)

In [ ]:
# Load CASIA dataset with perturbations applied
# Returns 6 values when return_perturbation_type=True:
#   X_train, X_test  — image arrays (object dtype, variable size PIL images)
#   y_train, y_test  — string labels (the Chinese character/sequence)
#   perturbation_type_train/test — None for clean samples, string label for perturbed
(
    X_train,
    X_test,
    y_train,
    y_test,
    perturbation_type_train,
    perturbation_type_test,
) = u.load_data(
    show_progress=True,
    perturb_count=None,           # auto-calculated from bucket settings
    include_combinations=False,    # use power-set combinations of perturbations
    min_per_perturb_bucket=50,    # at least 50 samples per perturbation bucket
    return_perturbation_type=True,
    save_npy=True,                    # save to .npy for faster loading next time
    npy_output_dir="data",
)

Applying perturbations: 100%|██████████| 450/450 [00:00<00:00, 7602.75it/s]


In [4]:
# Verify dataset dimensions: (n_samples,)
print("Train / Test sizes:")
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Train / Test sizes:
(1560,) (390,) (1560,) (390,)


In [4]:
# Split train into train / validation (80 / 20)
X_train, X_val, y_train, y_val = u.split_data(X_train, y_train)
print("Train / Val sizes:")
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

Train / Val sizes:
(1248,) (312,) (1248,) (312,)


In [6]:
# Explore label frequency distribution across train / val / test
u.explore_data(X_train, y_train, y_test, y_val)

/Users/ek/Documents/GitHub/vr-chinese-ocr-eval/data/utils.py:359: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Perturbation Analysis

Visual validation that each perturbation type looks realistic for the Meta Quest whiteboard capture scenario.

**Grid layout:** rows = perturbation type, columns = sample images  
The first row is always the original clean image for reference.

In [5]:
# Import the perturbation analysis module from /perturbations/
from perturbations.perturbation_analysis import show_perturbation_grid

In [8]:
# Generate the visual analysis grid
# n_samples controls how many CASIA images are used as columns (keep <= 8 for readability)
# save_path saves the figure as a PNG alongside the notebook
show_perturbation_grid(images_raw=X_train, labels_raw=y_train,n_samples=5, save_path="perturbation_analysis.png")

  Applying: radial_glare ...
  Applying: light_streak ...
  Applying: gaussian_noise ...
  Applying: contrast_variation ...
  Applying: occlusion ...
  Applying: perspective_warp ...
  Applying: motion_blur ...
  Applying: white_balance_shift ...
  Applying: low_resolution ...

Figure saved to: perturbation_analysis.png

Done.


/Users/ek/Documents/GitHub/vr-chinese-ocr-eval/perturbations/perturbation_analysis.py:232: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Model Inference

Each model is run against the same test set (clean + perturbed).  
Results are stored per model so we can compare robustness across perturbation types in the next section.

Models:
- **CnOCR** — PyTorch-based, 20+ pretrained Chinese OCR models
- **ANCHOR** — VGG CNN built specifically for handwritten Chinese, 97% on ICDAR 2013
- **TrOCR** — Microsoft transformer model trained on handwritten and real-world text
- **DINOv3** — Meta self-supervised vision transformer (general-purpose baseline)

In [4]:
# Run model inference (CnOCR required; others optional)
from models.cnocr_inference import run_cnocr

predictions = {
    'cnocr': run_cnocr(X_test),
}

# Optional models: run if available, but do not fail the notebook if missing.
try:
    from models.anchor_inference import run_anchor
    predictions['anchor'] = run_anchor(X_test)
except Exception as e:
    print(f"Skipping anchor inference: {e}")

try:
    from models.trocr_inference import run_trocr
    predictions['trocr'] = run_trocr(X_test)
except Exception as e:
    print(f"Skipping trocr inference: {e}")

# DINOv3 integration is pending in this repo.



CnOCR inference: 100%|██████████| 390/390 [00:00<00:00, 539.51it/s]


[ANCHOR] WARNING: Weights not found at /Users/ek/anchor/data/weights08.h5
[ANCHOR] Clone https://github.com/angzhou/anchor to ~/anchor


Loading weights: 100%|██████████| 478/478 [00:00<00:00, 6393.69it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
TrOCR inference:   0%|          | 0/390 [00:00<?, ?it/s]/opt/homebrew/lib/python3.11/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
TrOCR inference: 100%|██████████| 390/390 [02:42<00:00,  2.39it/s]


NameError: name 'run_dinov3' is not defined

## 5. Robustness Evaluation

Track accuracy and recall per model as perturbation type changes.  
Goal: identify which environmental conditions (glare, perspective, motion blur, etc.) break each model most.

In [5]:
# CnOCR robustness evaluation: overall + per-perturbation metrics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evaluation.cnocr_evaluation import exact_match_accuracy, character_error_rate

# Reuse predictions from Section 4 when available.
if 'predictions' in globals() and 'cnocr' in predictions:
    y_pred_cnocr = np.asarray(predictions['cnocr'], dtype=object)
else:
    from evaluation.cnocr_evaluation import evaluate_cnocr
    eval_out = evaluate_cnocr(X_test, y_test, return_predictions=True)
    y_pred_cnocr = np.asarray(eval_out['predictions'], dtype=object)

y_true = np.asarray(y_test, dtype=object)
ptype = np.asarray(perturbation_type_test, dtype=object)

overall_exact = exact_match_accuracy(y_true, y_pred_cnocr)
overall_cer = character_error_rate(y_true, y_pred_cnocr)
print(f"CnOCR overall exact-match accuracy: {overall_exact:.4f}")
print(f"CnOCR overall CER: {overall_cer:.4f}")

def _label_perturbation(v):
    if v is None:
        return 'clean'
    if isinstance(v, float) and np.isnan(v):
        return 'clean'
    return str(v)

rows = []
for raw in pd.unique(ptype):
    mask = np.array([(_label_perturbation(x) == _label_perturbation(raw)) for x in ptype])
    if not mask.any():
        continue
    label = _label_perturbation(raw)
    rows.append({
        'perturbation_type': label,
        'n_samples': int(mask.sum()),
        'exact_match': exact_match_accuracy(y_true[mask], y_pred_cnocr[mask]),
        'cer': character_error_rate(y_true[mask], y_pred_cnocr[mask]),
    })

robustness_df = pd.DataFrame(rows).sort_values(['exact_match', 'n_samples'], ascending=[False, False])
display(robustness_df)

plt.figure(figsize=(10, 4))
plt.bar(robustness_df['perturbation_type'], robustness_df['exact_match'])
plt.xticks(rotation=40, ha='right')
plt.ylabel('Exact-match accuracy')
plt.title('CnOCR Robustness by Perturbation Type')
plt.tight_layout()
plt.show()



CnOCR inference: 100%|██████████| 390/390 [00:01<00:00, 327.74it/s]


CnOCR overall exact-match accuracy: 0.0026
CnOCR overall CER: 1.0077


,perturbation_type,n_samples,exact_match,cer
0,clean,292,0.003425,1.008009
8,light_streak,16,0.000000,1.000000
7,low_resolution,13,0.000000,1.051282
1,occlusion,11,0.000000,1.000000
2,contrast_variation,11,0.000000,1.000000
3,radial_glare,11,0.000000,1.000000
4,gaussian_noise,10,0.000000,1.000000
5,perspective_warp,9,0.000000,1.000000
6,white_balance_shift,9,0.000000,1.000000
9,motion_blur,8,0.000000,1.000000


/var/folders/24/hvx26y8s46n0dw1s62vn4k3m0000gn/T/ipykernel_98697/3794243819.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
print(predictions['cnocr'][:5])


NameError: name 'predictions' is not defined

: 

In [ ]:
print(y_test[:5])